<a href="https://colab.research.google.com/github/JonathanBastineKho/GLiNER_Modular_RAG/blob/main/gliner_build_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main Biomedical Corpora

I've chosen three corpora to build our biomedical retrieval knowledge base. All are expected to be high yield for their relatively small size:

1. MeSH aka Medical Subject Headings (~31.0k documents)
- Technical vocabulary used to index PubMed, which is a search engine for scientific abstracts related to biomedical literature
- Contains disease definitions, chemical descriptions, synonyms, hierarchical relations etc (e.g. “Diabetes Mellitus: A heterogeneous group of disorders characterized by hyperglycemia...”)
2. Disease Ontology (~14.5k documents)
- Disease vocabulary
- Contains disease names, definitions, synonyms and cross-references
3. DrugCentral (~4.1k documents)
- Open-access online drug information repository
- Contains drug names, mechanisms, targets etc

**Total: about 49.6k documents in 33.3k chunks**
- Number of chunks < documents, because very short documents are discarded

-----
Side note: I realise that other biomedical RAG pipeline projects may use more extensive corpora such as PubMed and/or PMC Open Access subset. These contain the abstracts and full papers of hundreds of thousands of journal articles. I've chosen NOT to use these for two reasons:
1. File size is expected to be massive and downloading everything will be a gargantuan task.
2. Risk of data leakage - some of the evaluation datasets (e.g. NCBI Disease, BC5DR) are explicitly collections of PubMed articles/abstracts

In [1]:
def build_biomedical_rag_corpus(output_dir="biomedical_rag"):
    """
    Build a RAG-ready biomedical corpus from:
      1) MeSH
      2) Disease Ontology
      3) DrugCentral

    Outputs:
      - mesh.jsonl
      - disease_ontology.jsonl
      - drugcentral.jsonl
      - biomedical_rag_corpus.jsonl
      - biomedical_rag_corpus_chunked.jsonl
    """

    import os
    import re
    import json
    import csv
    import gzip
    from collections import defaultdict

    import requests
    import xml.etree.ElementTree as ET

    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 biomedical-rag-builder/1.0"})

    def download(url, path, timeout=120):
        r = session.get(url, stream=True, timeout=timeout)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    def clean_text(text):
        if text is None:
            return ""
        text = re.sub(r"\s+", " ", str(text)).strip()
        return text

    def write_jsonl(records, path):
        with open(path, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    def chunk_text(text, size=200, overlap=40):
        words = text.split()
        if not words:
            return []
        step = max(1, size - overlap)
        chunks = []
        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + size]).strip()
            if len(chunk.split()) > 20:
                chunks.append(chunk)
            if i + size >= len(words):
                break
        return chunks

    def find_latest_mesh_url():
        index_url = "https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/"
        html = session.get(index_url, timeout=60).text

        matches = re.findall(r'href="(desc\d{4}\.xml)"', html, flags=re.IGNORECASE)
        if not matches:
            raise RuntimeError("Could not find a MeSH descriptor XML file in the MeSH directory listing.")

        latest_file = sorted(set(matches))[-1]
        return index_url + latest_file

    def find_drugcentral_links():
        """
        DrugCentral download page currently links:
        - drug.target.interaction.tsv.gz
        - structures.smiles.tsv
        We first try to scrape the current page. If that fails, we fall back to
        the currently-linked file paths.
        """
        page_url = "https://drugcentral.org/download"
        html = session.get(page_url, timeout=60).text

        tsv_match = re.search(
            r'href="(https?://[^"]*drug\.target\.interaction\.tsv\.gz|/[^"]*drug\.target\.interaction\.tsv\.gz)"',
            html,
            flags=re.IGNORECASE,
        )
        smiles_match = re.search(
            r'href="(https?://[^"]*structures\.smiles\.tsv|/[^"]*structures\.smiles\.tsv)"',
            html,
            flags=re.IGNORECASE,
        )

        def absolutize(href):
            if href.startswith("http://") or href.startswith("https://"):
                return href
            return "https://drugcentral.org" + href

        if tsv_match:
            tsv_url = absolutize(tsv_match.group(1))
        else:
            # Fallback based on current official page links
            tsv_url = "https://unmtid-dbs.net/download/DrugCentral/2021_09_01/drug.target.interaction.tsv.gz"

        if smiles_match:
            smiles_url = absolutize(smiles_match.group(1))
        else:
            # Fallback based on current official page links
            smiles_url = "https://unmtid-dbs.net/download/DrugCentral/2021_09_01/structures.smiles.tsv"

        return tsv_url, smiles_url

    all_records = []

    # ------------------------------------------------------------------
    # 1) MeSH
    # ------------------------------------------------------------------
    mesh_url = find_latest_mesh_url()
    mesh_xml = os.path.join(output_dir, os.path.basename(mesh_url))

    if not os.path.exists(mesh_xml):
        print(f"Downloading MeSH from {mesh_url} ...")
        download(mesh_url, mesh_xml)

    with open(mesh_xml, "r", encoding="utf-8", errors="ignore") as f:
        head = f.read(500)

    if "<DescriptorRecordSet" not in head and "<?xml" not in head:
        raise RuntimeError(
            f"{mesh_xml} does not look like valid MeSH XML. "
            "You may have downloaded an HTML error page instead."
        )

    mesh_records = []
    tree = ET.parse(mesh_xml)
    root = tree.getroot()

    for record in root.findall(".//DescriptorRecord"):
        name = clean_text(record.findtext(".//DescriptorName/String"))
        scope = clean_text(record.findtext(".//ScopeNote"))

        synonyms = []
        for term in record.findall(".//Term/String"):
            s = clean_text(term.text)
            if s and s != name:
                synonyms.append(s)

        text_parts = []
        if scope:
            text_parts.append(scope)
        if synonyms:
            text_parts.append("Synonyms: " + "; ".join(sorted(set(synonyms))[:20]))

        if name and text_parts:
            mesh_records.append({
                "source": "MeSH",
                "title": name,
                "text": clean_text(" ".join(text_parts))
            })

    write_jsonl(mesh_records, os.path.join(output_dir, "mesh.jsonl"))
    all_records.extend(mesh_records)

    # ------------------------------------------------------------------
    # 2) Disease Ontology
    # ------------------------------------------------------------------
    do_url = "https://purl.obolibrary.org/obo/doid.obo"
    do_file = os.path.join(output_dir, "doid.obo")

    if not os.path.exists(do_file):
        print("Downloading Disease Ontology ...")
        download(do_url, do_file)

    do_records = []
    with open(do_file, "r", encoding="utf-8", errors="ignore") as f:
        blocks = f.read().split("\n[Term]\n")

    for block in blocks:
        name = ""
        definition = ""
        synonyms = []

        for line in block.splitlines():
            if line.startswith("name:"):
                name = clean_text(line.replace("name:", "", 1))
            elif line.startswith("def:"):
                m = re.search(r'"(.*?)"', line)
                if m:
                    definition = clean_text(m.group(1))
            elif line.startswith("synonym:"):
                m = re.search(r'"(.*?)"', line)
                if m:
                    synonyms.append(clean_text(m.group(1)))

        if name:
            text_parts = []
            if definition:
                text_parts.append(definition)
            if synonyms:
                text_parts.append("Synonyms: " + "; ".join(sorted(set(synonyms))[:20]))

            do_records.append({
                "source": "DiseaseOntology",
                "title": name,
                "text": clean_text(" ".join(text_parts))
            })

    write_jsonl(do_records, os.path.join(output_dir, "disease_ontology.jsonl"))
    all_records.extend(do_records)

    # ------------------------------------------------------------------
    # 3) DrugCentral
    # ------------------------------------------------------------------
    import csv
    import gzip
    from collections import defaultdict

    drugcentral_records = []

    tsv_url, smiles_url = find_drugcentral_links()

    drug_tsv_gz = os.path.join(output_dir, "drugcentral_drug_target.tsv.gz")
    if not os.path.exists(drug_tsv_gz):
        print(f"Downloading DrugCentral drug-target TSV from {tsv_url} ...")
        download(tsv_url, drug_tsv_gz)

    smiles_file = os.path.join(output_dir, os.path.basename(smiles_url))
    if not os.path.exists(smiles_file):
        print(f"Downloading DrugCentral SMILES file from {smiles_url} ...")
        download(smiles_url, smiles_file)

    drug_map = defaultdict(lambda: {
        "targets": set(),
        "actions": set(),
        "genes": set(),
        "ids": set(),
    })

    # --- inspect and parse TSV defensively ---
    with gzip.open(drug_tsv_gz, "rt", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f, delimiter="\t")
        headers = reader.fieldnames or []
        print("DrugCentral TSV headers:", headers)

        def first_present(row, candidates):
            for c in candidates:
                if c in row and row[c]:
                    return clean_text(row[c])
            return ""

        for row in reader:
            drug_name = first_present(row, [
                "drug_name", "name", "inn", "ingredient_name",
                "drug", "drug_name_standardized", "drugname"
            ])
            if not drug_name:
                continue

            target = first_present(row, [
                "target_name", "target", "gene_name", "target_gene_name"
            ])
            action = first_present(row, [
                "action_type", "action", "interaction_type", "moa"
            ])
            gene = first_present(row, [
                "gene", "gene_symbol", "target_gene_symbol", "symbol"
            ])
            dc_id = first_present(row, [
                "struct_id", "drugcentral_id", "id"
            ])

            if target:
                drug_map[drug_name]["targets"].add(target)
            if action:
                drug_map[drug_name]["actions"].add(action)
            if gene:
                drug_map[drug_name]["genes"].add(gene)
            if dc_id:
                drug_map[drug_name]["ids"].add(dc_id)

    # --- parse SMILES file and use it as fallback / enrichment ---
    smiles_meta = defaultdict(dict)

    with open(smiles_file, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f, delimiter="\t")
        smiles_headers = reader.fieldnames or []
        print("DrugCentral SMILES headers:", smiles_headers)

        def first_present(row, candidates):
            for c in candidates:
                if c in row and row[c]:
                    return clean_text(row[c])
            return ""

        for row in reader:
            drug_name = first_present(row, ["INN", "drug_name", "name", "inn"])
            if not drug_name:
                continue

            smiles = first_present(row, ["SMILES", "smiles"])
            inchi = first_present(row, ["InChI", "inchi"])
            cas = first_present(row, ["CAS_RN", "CAS", "cas"])
            dc_id = first_present(row, ["ID", "id", "struct_id", "drugcentral_id"])

            if smiles:
                smiles_meta[drug_name]["smiles"] = smiles
            if inchi:
                smiles_meta[drug_name]["inchi"] = inchi
            if cas:
                smiles_meta[drug_name]["cas"] = cas
            if dc_id:
                smiles_meta[drug_name]["drugcentral_id"] = dc_id

    # If the drug-target TSV gave nothing, fall back to SMILES file alone
    all_drug_names = set(drug_map.keys()) | set(smiles_meta.keys())

    for drug_name in sorted(all_drug_names):
        info = drug_map.get(drug_name, {
            "targets": set(), "actions": set(), "genes": set(), "ids": set()
        })

        text_parts = [drug_name]
        if info["targets"]:
            text_parts.append("Targets: " + "; ".join(sorted(info["targets"])[:20]))
        if info["actions"]:
            text_parts.append("Actions: " + "; ".join(sorted(info["actions"])[:10]))
        if info["genes"]:
            text_parts.append("Genes: " + "; ".join(sorted(info["genes"])[:10]))

        meta = {}
        if info["ids"]:
            meta["drugcentral_ids"] = sorted(info["ids"])[:10]
        if drug_name in smiles_meta:
            meta.update(smiles_meta[drug_name])

        drugcentral_records.append({
            "source": "DrugCentral",
            "title": drug_name,
            "text": clean_text(" ".join(text_parts)),
            "meta": meta
        })

    write_jsonl(drugcentral_records, os.path.join(output_dir, "drugcentral.jsonl"))
    all_records.extend(drugcentral_records)

    # ------------------------------------------------------------------
    # 4) Merge corpus
    # ------------------------------------------------------------------
    merged = os.path.join(output_dir, "biomedical_rag_corpus.jsonl")
    write_jsonl(all_records, merged)

    # ------------------------------------------------------------------
    # 5) Chunk corpus
    # ------------------------------------------------------------------
    chunk_file = os.path.join(output_dir, "biomedical_rag_corpus_chunked.jsonl")
    chunks = []

    for i, doc in enumerate(all_records):
        for j, c in enumerate(chunk_text(doc["text"])):
            chunks.append({
                "doc_id": i,
                "chunk_id": f"{i}_{j}",
                "source": doc["source"],
                "title": doc["title"],
                "text": c,
                "meta": doc.get("meta", {})
            })

    write_jsonl(chunks, chunk_file)

    return {
        "documents": len(all_records),
        "chunks": len(chunks),
        "n_mesh": len(mesh_records),
        "n_disease_ontology": len(do_records),
        "n_drugcentral": len(drugcentral_records),
        "mesh_jsonl": os.path.join(output_dir, "mesh.jsonl"),
        "disease_ontology_jsonl": os.path.join(output_dir, "disease_ontology.jsonl"),
        "drugcentral_jsonl": os.path.join(output_dir, "drugcentral.jsonl"),
        "merged_jsonl": merged,
        "chunked_jsonl": chunk_file,
        "output_dir": output_dir,
    }

In [2]:
# Test case -- this works as of 7 Mar 2026 (Wilbert)

result = build_biomedical_rag_corpus("biomedical_rag")
print(result)

DrugCentral TSV headers: ['DRUG_NAME', 'STRUCT_ID', 'TARGET_NAME', 'TARGET_CLASS', 'ACCESSION', 'GENE', 'SWISSPROT', 'ACT_VALUE', 'ACT_UNIT', 'ACT_TYPE', 'ACT_COMMENT', 'ACT_SOURCE', 'RELATION', 'MOA', 'MOA_SOURCE', 'ACT_SOURCE_URL', 'MOA_SOURCE_URL', 'ACTION_TYPE', 'TDL', 'ORGANISM']
DrugCentral SMILES headers: ['SMILES', 'InChI', 'InChIKey', 'ID', 'INN', 'CAS_RN']
{'documents': 49681, 'chunks': 33359, 'n_mesh': 31050, 'n_disease_ontology': 14532, 'n_drugcentral': 4099, 'mesh_jsonl': 'biomedical_rag/mesh.jsonl', 'disease_ontology_jsonl': 'biomedical_rag/disease_ontology.jsonl', 'drugcentral_jsonl': 'biomedical_rag/drugcentral.jsonl', 'merged_jsonl': 'biomedical_rag/biomedical_rag_corpus.jsonl', 'chunked_jsonl': 'biomedical_rag/biomedical_rag_corpus_chunked.jsonl', 'output_dir': 'biomedical_rag'}


In [3]:
! ls -lh /content/biomedical_rag

total 359M
-rw-r--r-- 1 root root  17M Mar  9 07:45 biomedical_rag_corpus_chunked.jsonl
-rw-r--r-- 1 root root  18M Mar  9 07:45 biomedical_rag_corpus.jsonl
-rw-r--r-- 1 root root 299M Mar  9 07:44 desc2026.xml
-rw-r--r-- 1 root root 3.8M Mar  9 07:45 disease_ontology.jsonl
-rw-r--r-- 1 root root 6.7M Mar  9 07:45 doid.obo
-rw-r--r-- 1 root root 762K Mar  9 07:45 drugcentral_drug_target.tsv.gz
-rw-r--r-- 1 root root 1.5M Mar  9 07:45 drugcentral.jsonl
-rw-r--r-- 1 root root  13M Mar  9 07:45 mesh.jsonl
-rw-r--r-- 1 root root 1.1M Mar  9 07:45 structures.smiles.tsv


# Supplementary Anatomical Corpora

I've chosen three supplemental corpora to expand our anatomical retrieval knowledge base.

1. Uberon aka Multi-species Anatomy Ontology (~27.0k documents)

- Cross-species anatomical ontology representing structures, functions, and developmental lineages

- Contains anatomical definitions, synonyms, and hierarchical part-of relationships (e.g., “cerebral cortex: The gray matter of the surface of the cerebral hemisphere...”)

2. Human Phenotype Ontology (HPO) (~19.0k documents)

- Standardized vocabulary of phenotypic abnormalities encountered in human disease

- Contains clinical signs, symptoms, definitions, and synonyms used for computational differential diagnostics

3. Cell Ontology (CL) (~3.8k documents)

- Standardized vocabulary for describing and classifying cell types across biology

- Contains microscopic anatomical definitions, synonyms, and cell lineage relationships


Total: about 49.8k documents in ~34.0k chunks

Note: I agree that using PubMed datasets will result in data leakage. Our target evaluation dataset (AnatEM) is explicitly a collection of PubMed and PMC articles. Including datasets like CRAFT or MedMentions creates a significant risk that our model will inadvertently retrieve the exact test sentences during evaluation, invalidating our results. Our combined RAG database is now leakage-free.

In [4]:
def build_biomedical_rag_corpus_2(output_dir="biomedical_rag_2"):
    """
    Build a supplemental RAG-ready biomedical corpus from:
      1) Uberon (Multi-species Anatomy Ontology)
      2) Human Phenotype Ontology (HPO)
      3) Cell Ontology (CL)

    Outputs:
      - uberon.jsonl
      - hpo.jsonl
      - cell_ontology.jsonl
      - biomedical_rag_corpus_2.jsonl
      - biomedical_rag_corpus_2_chunked.jsonl
    """

    import os
    import re
    import json
    import requests

    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 biomedical-rag-builder-2/1.0"})

    def download(url, path, timeout=120):
        r = session.get(url, stream=True, timeout=timeout)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    def clean_text(text):
        if text is None:
            return ""
        text = re.sub(r"\s+", " ", str(text)).strip()
        return text

    def write_jsonl(records, path):
        with open(path, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    def chunk_text(text, size=200, overlap=40):
        words = text.split()
        if not words:
            return []
        step = max(1, size - overlap)
        chunks = []
        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + size]).strip()
            if len(chunk.split()) > 20:
                chunks.append(chunk)
            if i + size >= len(words):
                break
        return chunks

    def parse_obo_file(file_path, source_name):
        """Reuses Wilbert's OBO parsing logic for uniform extraction."""
        records = []
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            # Split by [Term] blocks
            blocks = f.read().split("\n[Term]\n")

        # Skip the first block (it's just the file header metadata)
        for block in blocks[1:]:
            name = ""
            definition = ""
            synonyms = []

            for line in block.splitlines():
                if line.startswith("name:"):
                    name = clean_text(line.replace("name:", "", 1))
                elif line.startswith("def:"):
                    m = re.search(r'"(.*?)"', line)
                    if m:
                        definition = clean_text(m.group(1))
                elif line.startswith("synonym:"):
                    m = re.search(r'"(.*?)"', line)
                    if m:
                        synonyms.append(clean_text(m.group(1)))

            if name:
                text_parts = []
                if definition:
                    text_parts.append(definition)
                if synonyms:
                    text_parts.append("Synonyms: " + "; ".join(sorted(set(synonyms))[:20]))

                # Only append if we actually have text beyond just the name
                if text_parts:
                    records.append({
                        "source": source_name,
                        "title": name,
                        "text": clean_text(" ".join(text_parts))
                    })
        return records

    all_records = []

    # Define our 3 new datasets and their official PURL download links
    datasets = [
        ("Uberon", "http://purl.obolibrary.org/obo/uberon.obo", "uberon.obo", "uberon.jsonl"),
        ("HPO", "http://purl.obolibrary.org/obo/hp.obo", "hp.obo", "hpo.jsonl"),
        ("CellOntology", "http://purl.obolibrary.org/obo/cl.obo", "cl.obo", "cell_ontology.jsonl")
    ]

    counts = {}

    # 1) Download and Parse each OBO file
    for source_name, url, obo_filename, jsonl_filename in datasets:
        obo_path = os.path.join(output_dir, obo_filename)
        jsonl_path = os.path.join(output_dir, jsonl_filename)

        if not os.path.exists(obo_path):
            print(f"Downloading {source_name} from {url} ...")
            download(url, obo_path)

        print(f"Parsing {source_name}...")
        records = parse_obo_file(obo_path, source_name)

        write_jsonl(records, jsonl_path)
        all_records.extend(records)
        counts[source_name] = len(records)

    # 2) Merge corpus
    merged = os.path.join(output_dir, "biomedical_rag_corpus_2.jsonl")
    write_jsonl(all_records, merged)

    # 3) Chunk corpus
    chunk_file = os.path.join(output_dir, "biomedical_rag_corpus_2_chunked.jsonl")
    chunks = []

    for i, doc in enumerate(all_records):
        for j, c in enumerate(chunk_text(doc["text"])):
            chunks.append({
                "doc_id": i,
                "chunk_id": f"{i}_{j}",
                "source": doc["source"],
                "title": doc["title"],
                "text": c,
                "meta": {}  # OBO files don't have complex nested meta, keeping it uniform
            })

    write_jsonl(chunks, chunk_file)

    return {
        "documents": len(all_records),
        "chunks": len(chunks),
        "counts": counts,
        "merged_jsonl": merged,
        "chunked_jsonl": chunk_file,
        "output_dir": output_dir,
    }

In [5]:
result = build_biomedical_rag_corpus_2("biomedical_rag_2")
print("\n--- Generation Complete ---")
print(result)

Parsing Uberon...
Parsing HPO...
Parsing CellOntology...

--- Generation Complete ---
{'documents': 61016, 'chunks': 35326, 'counts': {'Uberon': 24991, 'HPO': 18222, 'CellOntology': 17803}, 'merged_jsonl': 'biomedical_rag_2/biomedical_rag_corpus_2.jsonl', 'chunked_jsonl': 'biomedical_rag_2/biomedical_rag_corpus_2_chunked.jsonl', 'output_dir': 'biomedical_rag_2'}


# Irrelevant Corpora (Programming Domain)

This irrelevant retrieval knowledge base is built from programming documentation. I think it's a good choice to use this as one "irrelevant domain" since the docs for common libraries are readily available online and downloadable.

1. Python (~0.5k documents)
2. NumPy (~2.7k documents)
3. SciPy (~4.7k documents)
4. Pandas (~2.3k documents)
5. Django (~0.6k documents)
6. scikit-learn (~1.0k documents)

**Total: about 12.1k documents in 66.1k chunks**

All six are official programming/software-documentation sources. The official sites expose downloadable HTML or offline HTML packages for Python, pandas, NumPy/SciPy and Django.

In [6]:
def build_irrelevant_programming_rag_corpus(output_dir="irrelevant_programming_rag"):
    """
    Build an intentionally irrelevant RAG corpus from software/programming docs:
      1) Python
      2) NumPy
      3) SciPy
      4) pandas
      5) Django
      6) scikit-learn

    Outputs:
      - one JSONL per source
      - merged document-level JSONL
      - merged chunk-level JSONL
    """

    import os
    import re
    import json
    import zipfile
    import requests
    from html import unescape

    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 irrelevant-rag-builder/1.0"})

    def download(url, path, timeout=180):
        r = session.get(url, stream=True, timeout=timeout)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    def clean_text(text):
        if text is None:
            return ""
        text = unescape(text)
        text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
        text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
        text = re.sub(r"(?is)<[^>]+>", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_title(html_text, fallback):
        m = re.search(r"(?is)<title>(.*?)</title>", html_text)
        if m:
            title = clean_text(m.group(1))
            if title:
                return title
        return fallback

    def write_jsonl(records, path):
        with open(path, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    def chunk_text(text, size=220, overlap=40):
        words = text.split()
        if not words:
            return []
        step = max(1, size - overlap)
        chunks = []
        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + size]).strip()
            if len(chunk.split()) >= 30:
                chunks.append(chunk)
            if i + size >= len(words):
                break
        return chunks

    def collect_html_docs(root_dir, source_name):
        records = []
        for dirpath, _, filenames in os.walk(root_dir):
            for fn in filenames:
                if not fn.lower().endswith(".html"):
                    continue
                path = os.path.join(dirpath, fn)
                try:
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        raw = f.read()
                except Exception:
                    continue

                text = clean_text(raw)
                if len(text.split()) < 50:
                    continue

                relpath = os.path.relpath(path, root_dir)
                title = get_title(raw, relpath)

                records.append({
                    "source": source_name,
                    "title": title,
                    "text": text,
                    "meta": {"path": relpath}
                })
        return records

    def scrape_first_match(url, pattern):
        html = session.get(url, timeout=60).text
        m = re.search(pattern, html, flags=re.IGNORECASE)
        if not m:
            raise RuntimeError(f"Could not find download link from {url}")
        href = m.group(1)
        if href.startswith("http"):
            return href
        # root-relative
        if href.startswith("/"):
            base = re.match(r"^(https?://[^/]+)", url).group(1)
            return base + href
        # page-relative
        return url.rsplit("/", 1)[0] + "/" + href

    # ------------------------------------------------------------------
    # Discover archive URLs from official pages
    # ------------------------------------------------------------------

    # Python docs: official download page has HTML zip
    python_page = "https://docs.python.org/3/download.html"
    python_zip_url = scrape_first_match(
        python_page,
        r'href="([^"]+python-[^"]+-docs-html\.zip|[^"]+python-[^"]+docs-html\.zip|[^"]+html\.zip)"'
    )

    # pandas docs: official page exposes "Zipped HTML"
    pandas_page = "https://pandas.pydata.org/docs/"
    pandas_zip_url = scrape_first_match(
        pandas_page,
        r'href="([^"]*pandas\.zip)"'
    )

    # NumPy docs: SciPy docs page exposes Complete Numpy Manual [HTML+zip]
    numpy_zip_url = "https://docs.scipy.org/doc/numpy/numpy-html.zip"

    # SciPy docs: pick a stable release HTML+zip directly from the SciPy docs host
    # Using 1.16.2 here as a concrete stable release.
    scipy_zip_url = "https://docs.scipy.org/doc/scipy-1.16.2/scipy-html-1.16.2.zip"

    # Django docs: official docs page exposes offline HTML zip
    django_page = "https://docs.djangoproject.com/en/6.0/"
    django_zip_url = scrape_first_match(
        django_page,
        r'href="(https://media\.djangoproject\.com/docs/django-docs-[^"]+\.zip|https://media\.django\.com/docs/django-docs-[^"]+\.zip|https://media\.djangoproject\.com/docs/[^"]+\.zip|https://media\.django\.com/docs/[^"]+\.zip|https://media\.djangoproject\.com/media/docs/[^"]+\.zip|https://media\.django\.com/media/docs/[^"]+\.zip|https://media\.djangoproject\.com/[^"]+\.zip|https://media\.django\.com/[^"]+\.zip|/[^"]*django-docs-[^"]+\.zip)"'
    )

    # scikit-learn docs: stable docs zip
    sklearn_zip_url = "https://scikit-learn.org/stable/_downloads/scikit-learn-docs.zip"

    sources = [
        ("PythonDocs", python_zip_url, "python_docs.zip", "python_docs"),
        ("NumPyDocs", numpy_zip_url, "numpy_docs.zip", "numpy_docs"),
        ("SciPyDocs", scipy_zip_url, "scipy_docs.zip", "scipy_docs"),
        ("PandasDocs", pandas_zip_url, "pandas_docs.zip", "pandas_docs"),
        ("DjangoDocs", django_zip_url, "django_docs.zip", "django_docs"),
        ("ScikitLearnDocs", sklearn_zip_url, "sklearn_docs.zip", "sklearn_docs"),
    ]

    all_records = []
    counts = {}

    for source_name, url, archive_name, extract_dir_name in sources:
        archive_path = os.path.join(output_dir, archive_name)
        extract_dir = os.path.join(output_dir, extract_dir_name)

        if not os.path.exists(archive_path):
            print(f"Downloading {source_name} from {url} ...")
            download(url, archive_path)

        if not os.path.exists(extract_dir):
            print(f"Extracting {source_name} ...")
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_dir)

        records = collect_html_docs(extract_dir, source_name)
        counts[source_name] = len(records)

        per_source_jsonl = os.path.join(output_dir, f"{source_name.lower()}.jsonl")
        write_jsonl(records, per_source_jsonl)
        all_records.extend(records)

    merged_jsonl = os.path.join(output_dir, "programming_rag_corpus.jsonl")
    write_jsonl(all_records, merged_jsonl)

    chunked_jsonl = os.path.join(output_dir, "programming_rag_corpus_chunked.jsonl")
    chunk_count = 0
    with open(chunked_jsonl, "w", encoding="utf-8") as f:
        for doc_id, rec in enumerate(all_records):
            for chunk_id, chunk in enumerate(chunk_text(rec["text"])):
                out = {
                    "doc_id": doc_id,
                    "chunk_id": f"{doc_id}_{chunk_id}",
                    "source": rec["source"],
                    "title": rec["title"],
                    "text": chunk,
                    "meta": rec.get("meta", {})
                }
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
                chunk_count += 1

    return {
        "documents": len(all_records),
        "chunks": chunk_count,
        "counts": counts,
        "merged_jsonl": merged_jsonl,
        "chunked_jsonl": chunked_jsonl,
        "output_dir": output_dir,
    }

In [7]:
# Test case -- this works as of 7 Mar 2026 (Wilbert)

result = build_irrelevant_programming_rag_corpus("irrelevant_programming_rag")
print(result)

Extracting PythonDocs ...
Extracting NumPyDocs ...
Extracting SciPyDocs ...
Extracting PandasDocs ...
Extracting DjangoDocs ...
Extracting ScikitLearnDocs ...
{'documents': 12138, 'chunks': 66131, 'counts': {'PythonDocs': 569, 'NumPyDocs': 2734, 'SciPyDocs': 4752, 'PandasDocs': 2371, 'DjangoDocs': 657, 'ScikitLearnDocs': 1055}, 'merged_jsonl': 'irrelevant_programming_rag/programming_rag_corpus.jsonl', 'chunked_jsonl': 'irrelevant_programming_rag/programming_rag_corpus_chunked.jsonl', 'output_dir': 'irrelevant_programming_rag'}


In [8]:
! ls -lh /content/irrelevant_programming_rag/

total 584M
drwxr-xr-x 13 root root 4.0K Mar  9 07:46 django_docs
-rw-r--r--  1 root root 5.5M Mar  9 07:46 djangodocs.jsonl
-rw-r--r--  1 root root 7.5M Mar  9 07:46 django_docs.zip
drwxr-xr-x 12 root root 4.0K Mar  9 07:46 numpy_docs
-rw-r--r--  1 root root  13M Mar  9 07:46 numpydocs.jsonl
-rw-r--r--  1 root root  34M Mar  9 07:46 numpy_docs.zip
drwxr-xr-x 12 root root 4.0K Mar  9 07:46 pandas_docs
-rw-r--r--  1 root root  38M Mar  9 07:46 pandasdocs.jsonl
-rw-r--r--  1 root root  38M Mar  9 07:46 pandas_docs.zip
-rw-r--r--  1 root root 131M Mar  9 07:47 programming_rag_corpus_chunked.jsonl
-rw-r--r--  1 root root 101M Mar  9 07:47 programming_rag_corpus.jsonl
drwxr-xr-x  3 root root 4.0K Mar  9 07:46 python_docs
-rw-r--r--  1 root root  12M Mar  9 07:46 pythondocs.jsonl
-rw-r--r--  1 root root  16M Mar  9 07:46 python_docs.zip
-rw-r--r--  1 root root  20M Mar  9 07:46 scikitlearndocs.jsonl
drwxr-xr-x 11 root root 4.0K Mar  9 07:46 scipy_docs
-rw-r--r--  1 root root  14M Mar  9 07:46

# Vector Store with ChromaDB


In [9]:
!pip install chromadb sentence-transformers


import os
import json
import chromadb
from chromadb.utils import embedding_functions
from tqdm import tqdm

# Setup database path on Google Colab
#DB_PATH = "/content/local_vector_store/"

# Leakage-free corpora on Colab
#MAIN_BIO_PATH = "/content/biomedical_rag/biomedical_rag_corpus_chunked.jsonl"
#SUPP_BIO_PATH = "/content/biomedical_rag_2/biomedical_rag_corpus_2_chunked.jsonl"
#NOISE_PROG_PATH = "/content/irrelevant_programming_rag/programming_rag_corpus_chunked.jsonl"

#relative paths for Github
DB_PATH = "../../data/vector_store/"

MAIN_BIO_PATH = "../../data/biomedical_rag_corpus_chunked.jsonl"
SUPP_BIO_PATH = "../../data/biomedical_rag_corpus_2_chunked.jsonl"
NOISE_PROG_PATH = "../../data/programming_rag_corpus_chunked.jsonl"


def process_jsonl_corpus(file_path, limit=None):
    """Function to load pre-chunked custom JSONL datasets and safely format metadata."""
    print(f"Loading custom corpus from {file_path}...")
    docs, metadatas, ids = [], [], []

    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}. Skipping.")
        return docs, metadatas, ids

    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            # If testing_mode is on, stop early
            if limit and i >= limit:
                break

            row = json.loads(line)

            # Extract text
            docs.append(row["text"])

            # Flatten the nested "meta" dictionary safely for ChromaDB
            meta_dict = {
                "source": row.get("source", "unknown"),
                "title": str(row.get("title", ""))[:100]  # Truncate title just in case it's massive
            }

            # String any extra metadata (like SMILES, InChI, path, arrays, etc.)
            for k, v in row.get("meta", {}).items():
                meta_dict[k] = str(v)

            metadatas.append(meta_dict)

            # Use the pre-computed chunk_id
            prefix = row.get("source", "doc").replace("/", "_").replace(" ", "")
            chunk_id = row.get("chunk_id", f"chk_{i}")
            ids.append(f"{prefix}_{chunk_id}")

    return docs, metadatas, ids


def build_biomedical_vector_store(testing_mode=False):
    print("Initializing ChromaDB client...")
    client = chromadb.PersistentClient(path=DB_PATH)

    # Using sentence transformer model
    sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

    collection = client.get_or_create_collection(
        name="biomedical_knowledge_base",
        embedding_function=sentence_transformer_ef,
        metadata={"hnsw:space": "cosine"}
    )

    all_docs, all_metadatas, all_ids = [], [], []

    # Limit to 500 chunks per dataset if testing_mode is True
    jsonl_limit = 500 if testing_mode else None

    try:
        # 1. Main Biomedical Dictionary (MeSH, Disease Ontology, DrugCentral)
        d, m, i = process_jsonl_corpus(MAIN_BIO_PATH, limit=jsonl_limit)
        all_docs.extend(d); all_metadatas.extend(m); all_ids.extend(i)

        # 2. Supplementary Anatomical Dictionary (Uberon, HPO, Cell Ontology)
        d, m, i = process_jsonl_corpus(SUPP_BIO_PATH, limit=jsonl_limit)
        all_docs.extend(d); all_metadatas.extend(m); all_ids.extend(i)

        # 3. Adversarial Programming Corpus (Python, NumPy, Pandas, etc.)
        d, m, i = process_jsonl_corpus(NOISE_PROG_PATH, limit=jsonl_limit)
        all_docs.extend(d); all_metadatas.extend(m); all_ids.extend(i)

    except Exception as e:
        print(f"\nError loading datasets. Details: {e}")
        return

    # Push to ChromaDB in batches
    batch_size = 5000
    total_docs = len(all_docs)
    print(f"\nTotal documents ready for ingestion: {total_docs}")

    if total_docs == 0:
        print("No documents were loaded. Ensure JSONL paths are correct.")
        return

    print("Embedding and adding to ChromaDB...")

    # tqdm will track the ~27 massive batches and calculate ETA
    for idx in tqdm(range(0, total_docs, batch_size), desc="Ingesting Batches"):
        end_idx = min(idx + batch_size, total_docs)
        collection.add(
            documents=all_docs[idx:end_idx],
            metadatas=all_metadatas[idx:end_idx],
            ids=all_ids[idx:end_idx]
        )

    print(f"\nSuccess! Full offline vector database compiled at {DB_PATH}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [10]:
#testing_mode=False when ready to build full database
build_biomedical_vector_store(testing_mode=False)


Initializing ChromaDB client...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading custom corpus from /content/biomedical_rag/biomedical_rag_corpus_chunked.jsonl...
Loading custom corpus from /content/biomedical_rag_2/biomedical_rag_corpus_2_chunked.jsonl...
Loading custom corpus from /content/irrelevant_programming_rag/programming_rag_corpus_chunked.jsonl...

Total documents ready for ingestion: 134816
Embedding and adding to ChromaDB...


Ingesting Batches: 100%|██████████| 27/27 [3:13:51<00:00, 430.79s/it]



Success! Full offline vector database compiled at /content/local_vector_store/


# Save Vector Store to Shared Google Drive

In [14]:
#Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/NLP_Project/vector_store/"

!cp -r /content/local_vector_store/* "/content/drive/MyDrive/NLP_Project/vector_store/"

print("Successfully copied to Google Drive!")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully copied to Google Drive!
